In [2]:
import timm
import torch
import config
from data_preproc.data_preproc_functions import Logger
from models.MedViT import MedViT_small, MedViT_base, MedViT_large
from misc import get_model_summary

logger=Logger()

config.device = torch.device('cpu')

channels = 3
depth = 96
height = 96
width = 96
n_features = len(config.features_dl)

In [4]:
from models.convnext import ConvNeXt, convnext_tiny, convnext_small, convnext_base, convnext_xlarge

#model = convnext_small(config, n_features)
model = ConvNeXt(config, n_features, depths=[1,3,1], dims=[128, 256, 512], kernel_size=3)

total_params = get_model_summary(config=config, model=model, input_size=[(config.batch_size, channels, depth, height, width), (config.batch_size, max(n_features, 1))], 
                                                device=config.device, logger=logger, save_to_file=False)

[3, 3, 27, 3]
4
[256, 512, 1024, 2048]
Layer (type (var_name))                                 Input Shape               Output Shape              Param #                   Kernel Shape
ConvNeXt (ConvNeXt)                                     [8, 3, 96, 96, 96]        [8, 1]                    --                        --
├─ModuleList (downsample_layers)                        --                        --                        (recursive)               --
│    └─Sequential (0)                                   [8, 3, 96, 96, 96]        [8, 256, 24, 24, 24]      --                        --
│    │    └─Conv3d (0)                                  [8, 3, 96, 96, 96]        [8, 256, 24, 24, 24]      49,408                    [4, 4, 4]
│    │    └─LayerNorm (1)                               [8, 256, 24, 24, 24]      [8, 256, 24, 24, 24]      512                       --
├─ModuleList (stages)                                   --                        --                        (recursive)   

In [5]:
kernel_size=7
kernel_size // 2

3

In [2]:
from models.swin_3D import SwinTransformer3D

model = SwinTransformer3D(config, n_features, patch_size=4, window_size=8, depths=[1,1,3,1], num_heads=[3, 6, 12, 24], embed_dim=int(96/2))

total_params = get_model_summary(config=config, model=model, input_size=[(config.batch_size, channels, depth, height, width), (config.batch_size, max(n_features, 1))], 
                                                device=config.device, logger=logger, save_to_file=False)

c:\Users\danie\anaconda3\envs\MedMNIST\Lib\site-packages\torch\functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3527.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
c:\Users\danie\anaconda3\envs\MedMNIST\Lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


Layer (type (var_name))                                 Input Shape               Output Shape              Param #                   Kernel Shape
SwinTransformer3D (SwinTransformer3D)                   [8, 3, 96, 96, 96]        [8, 1]                    --                        --
├─PatchEmbed3D (patch_embed)                            [8, 3, 96, 96, 96]        [8, 48, 24, 24, 24]       --                        --
│    └─Conv3d (proj)                                    [8, 3, 96, 96, 96]        [8, 48, 24, 24, 24]       9,264                     [4, 4, 4]
├─Dropout (pos_drop)                                    [8, 48, 24, 24, 24]       [8, 48, 24, 24, 24]       --                        --
├─ModuleList (layers)                                   --                        --                        --                        --
│    └─BasicLayer (0)                                   [8, 48, 24, 24, 24]       [8, 96, 12, 12, 12]       --                        --
│    │    └─ModuleList (

In [37]:
import torch
import torch.nn.functional as F

# Example tensor of size [8, 3, 3, 3, 384]
input_tensor = torch.randn(8, 3, 3, 3, 384)

# Calculate padding amounts
padding = (0, 0,
           1, 0,
           1, 0,
           1, 0)  # (L, R, T, B, F, B) for 5D input

# Apply padding
output_tensor = F.pad(input_tensor, padding)

print(output_tensor.size())  # Should output torch.Size([8, 4, 4, 4, 384])

torch.Size([8, 4, 4, 4, 384])


In [33]:
import torch

# Original tensor
original_tensor = torch.randn(8, 3, 3, 3, 384)

# New dimensions after padding
new_dims = (8, 4, 4, 4, 384)

# Calculate the padding needed for each dimension
padding_sizes = [(0, 1)] * 5  # Padding for each dimension: (L, R)

# Create a list to hold the concatenated tensors
concatenated_tensors = []

# Iterate over the original tensor's dimensions
for i in range(len(original_tensor.shape)):
    # Check if padding is needed for this dimension
    if padding_sizes[i][0] > 0 or padding_sizes[i][1] > 0:
        # Create a tensor filled with zeros for padding
        padding_tensor = torch.zeros(padding_sizes[i][0], new_dims[i + 1], new_dims[i + 2], new_dims[i + 3], new_dims[i + 4], device=original_tensor.device)
        # Concatenate the original tensor slice and the padding tensor along the current dimension
        concatenated_tensors.append(torch.cat((original_tensor[:, :, :, :, i::new_dims[i]], padding_tensor), dim=i))
    else:
        concatenated_tensors.append(original_tensor)

# Concatenate all slices along the first dimension
result_tensor = torch.stack(concatenated_tensors, dim=0)

print(result_tensor.shape)  # Should output torch.Size([8, 4, 4, 4, 384])

RuntimeError: Sizes of tensors must match except in dimension 0. Expected size 3 but got size 4 for tensor number 1 in the list.

In [35]:
# Assuming your tensor is named `x`
x = torch.randn(8, 3, 3, 3, 384)

# Define the padding for each dimension
# Padding format: (dim5_left, dim5_right, dim4_left, dim4_right, ..., dim1_left, dim1_right)
padding = (0, 0,  # No padding for the 5th dimension (size 384)
           1, 0,  # Pad 1 element at the beginning of the 4th dimension (depth), none at the end
           1, 0,  # Pad 1 element at the beginning of the 3rd dimension (height), none at the end
           1, 0)  # Pad 1 element at the beginning of the 2nd dimension (width), none at the end

# Apply the padding
x_padded = F.pad(x, padding)

print(x_padded.shape)

torch.Size([8, 4, 4, 4, 384])


In [2]:

# model = MedViT_small(config=config, n_features=n_features)
# #model = MedViT_base(config=config, n_features=n_features)
# model.to(config.device)

# total_params = get_model_summary(config=config, model=model, input_size=[(config.batch_size, channels, depth, height, width), (config.batch_size, max(n_features, 1))], 
#                                                 device=config.device, logger=logger, save_to_file=False)

In [4]:
from models.Swin import SwinTransformerV2

model = SwinTransformerV2(config=config, n_features=n_features, img_size=depth, in_chans=channels, embed_dim=32)
#model = MedViT_base(config=config, n_features=n_features)
#model.to(config.device)

total_params = get_model_summary(config=config, model=model, input_size=[(config.batch_size, channels, depth, height, width), (config.batch_size, max(n_features, 1))], 
                                                device=config.device, logger=logger, save_to_file=False)

IndexError: too many indices for tensor of dimension 2

In [4]:
1 * 3 * 7 * 3 * 7 * 3 * 7 * 1

9261

In [3]:
pip install torchinfo

Note: you may need to restart the kernel to use updated packages.
